In [2]:
!pip install google-generativeai
!pip install pypdf
!pip install pdf2image
!pip install pillow
!pip install python-dotenv

  Using cached google_ai_generativelanguage-0.6.15-py3-none-any.whl.metadata (5.7 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached googleapis_common_protos-1.70.0-py3-none-any.whl.metadata (9.3 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
  Using cached grpcio_status-1.73.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached httplib2-0.22.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached google_auth_httplib2-0.2.0-py2.py3-none-any.whl.metadata (2.2 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
Using ca

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

# Configure the Gemini API
genai.configure(api_key=api_key)

# VLM decisions

In [1]:
import os
import google.generativeai as genai
from PIL import Image
import base64
import io
import fitz  # PyMuPDF
import os
from dotenv import load_dotenv
import csv
import json
import shutil
import time
import gc
from pathlib import Path

# ---
# Your existing setup and functions (with improvements)
# ---
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

# Configure the Gemini API
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found. Please set it in your .env file.")
genai.configure(api_key=api_key)



model = genai.GenerativeModel(
    'gemini-2.5-pro',  # Changed to 1.5-pro for better stability
)

def convert_pdf_to_images(pdf_path, output_folder, dpi=120):  # Reduced DPI for faster processing
    """Converts each page of a PDF to a PNG image."""
    os.makedirs(output_folder, exist_ok=True)
    image_paths = []
    try:
        pdf_document = fitz.open(pdf_path)
        for page_number in range(len(pdf_document)):
            page = pdf_document[page_number]
            # Reduced resolution for faster processing and smaller file sizes
            pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
            output_path = os.path.join(output_folder, f"page_{page_number + 1}.png")
            pix.save(output_path)
            image_paths.append(output_path)
        pdf_document.close()
    except Exception as e:
        print(f"  [ERROR] Failed to convert PDF '{os.path.basename(pdf_path)}' to images: {e}")
        return []
    return image_paths

def safe_cleanup_images(image_paths, max_retries=3):
    """Safely cleanup image files with retry logic."""
    for image_path in image_paths:
        if os.path.exists(image_path):
            for attempt in range(max_retries):
                try:
                    os.remove(image_path)
                    break
                except PermissionError:
                    print(f"  [WARNING] Could not delete {image_path} (attempt {attempt + 1}), retrying...")
                    time.sleep(1)  # Wait a bit before retrying
                    gc.collect()  # Force garbage collection
                except Exception as e:
                    print(f"  [ERROR] Failed to delete {image_path}: {e}")
                    break

def safe_cleanup_folder(folder_path, max_retries=3):
    """Safely cleanup temporary folder with retry logic."""
    if os.path.exists(folder_path):
        for attempt in range(max_retries):
            try:
                shutil.rmtree(folder_path)
                break
            except PermissionError:
                print(f"  [WARNING] Could not delete folder {folder_path} (attempt {attempt + 1}), retrying...")
                time.sleep(2)
                gc.collect()
            except Exception as e:
                print(f"  [ERROR] Failed to delete folder {folder_path}: {e}")
                break

def ocr_with_gemini_retry(image_paths, instruction, max_retries=3, base_delay=5):
    """Processes images with Gemini OCR with retry logic for timeout errors."""
    
    # Close PIL images properly to avoid file locks
    images = []
    try:
        for path in image_paths:
            img = Image.open(path)
            # Convert to bytes to avoid file locks
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='PNG')
            img_byte_arr.seek(0)
            images.append(Image.open(img_byte_arr))
            img.close()  # Close the original file handle
    except Exception as e:
        print(f"  [ERROR] Failed to load images: {e}")
        return None
    
    prompt = f"""
    {instruction}
    
    These are pages from a PDF document. Extract all text content while preserving the structure.
    Pay special attention to tables, columns, headers, and any structured content.
    Maintain paragraph breaks and formatting.
    """
    
    for attempt in range(max_retries):
        try:
            print(f"  Attempt {attempt + 1} to process with Gemini...")
            response = model.generate_content([prompt, *images])
            return response.text
            
        except Exception as e:
            error_msg = str(e).lower()
            if "504" in error_msg or "deadline" in error_msg or "timeout" in error_msg:
                if attempt < max_retries - 1:
                    delay = base_delay * (2 ** attempt)  # Exponential backoff
                    print(f"  [WARNING] Timeout error (attempt {attempt + 1}). Retrying in {delay} seconds...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"  [ERROR] Max retries reached for timeout error: {e}")
                    return None
            elif "quota" in error_msg or "rate" in error_msg:
                if attempt < max_retries - 1:
                    delay = 60  # Wait longer for quota issues
                    print(f"  [WARNING] Rate limit/quota error. Waiting {delay} seconds...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"  [ERROR] Max retries reached for rate limit: {e}")
                    return None
            else:
                print(f"  [ERROR] Unexpected error: {e}")
                return None
    
    return None

def ocr_complex_document(image_paths):
    """Creates the specific prompt and calls the Gemini OCR function with retry."""
    instruction = """
        اقرأ النص القانوني أو حكم اللجنة المرفق بعناية تامة، واستخرج تفصيلاً كاملاً للقرار وفق الشكل التالي. التزم بالقواعد والتعليمات أدناه حرفياً.

        التزامات عامة:
        - استند فقط إلى النص الأصلي كما ورد في المستند؛ لا تضف أي تفسير، استنتاج، أو معلومات من خارج النص.
        - استخدم اللغة العربية الفصحى في كل الحقول.
        - لا تذكر أي مواد قانونية أو أرقام مواد/مراجع نظامية.
        - لا تذكر أي مبالغ مالية أو أرقام نقدية في أي حقل.
        - إذا كان أي حقل غير موجود صراحة في النص، اتركه كسلسلة فارغة "".
        - احتفظ بأسلوب قانوني ورسمي، لكن صِف بوضوح وبالتفصيل ما ورد في القرار عن وجهات النظر والأسباب.
        - إذا كان النص المستخرج غير واضح أو به غموض في المعنى، انسخه كما هو دون تعديل أو تفسير أو إعادة صياغة لتجنّب أي سوء فهم.
        - لا تضف أي نص خارج بنية JSON النهائية.

        هيكل الإخراج (أعد النتيجة بنفس البنية أدناه تمامًا):

        {
        "تفصيل_القرار": {
            "رقم_القرار_النهائي": "استخرج رقم القرار النهائي كما ورد نصاً. يكون موجود في رأس أول صفحة ويبدأ بـ IR أو VA.",
            "اسم_الدائرة_الابتدائية": "استخرج اسم دائرة الفصل الابتدائية كما ورد في النص.",
            "اسم_الدائرة_النهائية": "استخرج اسم الدائرة الاستئنافية أو اللجنة النهائية كما ورد في النص.",
            "اللجنة_الابتدائية": "سرد تفصيلي لقرار اللجنة الابتدائية كما ورد نصاً، متضمناً الأسباب والمبررات التي استندت إليها اللجنة ونتيجة القرار في هذه المرحلة.",
            "اللجنة_النهائية": "سرد تفصيلي لقرار اللجنة النهائية كما ورد نصاً، متضمناً الأسباب والمبررات التي استندت إليها اللجنة ونتيجة القرار في هذه المرحلة.",
            "البنود_محل_الدعوى": [
            {
                "اسم_البند": "انسخ اسم وعبارة البند كما وردت في القرار (عنوان البند).",
                "نبذة_مختصرة_عن_الاعتراض": "خلاصة وجيزة (جملة أو اثنتان) تشرح طبيعة الاعتراض على هذا البند كما وردت في القرار.",
                "وجهة_نظر_المكلف_بالتفصيل": "انقل بالتفصيل كل ما ورد في نص القرار عن دفوع المكلف/الشركة بهذا البند: الحجج، الوقائع والمستندات التي استند إليها، المطالبات والإشارات الزمنية أو العقدية أو المحاسبية أو الفنية التي ذكرها المكلف. انسخ النص قدر الإمكان مع إعادة ترتيب بسيط لقراءة منطقية إذا لزم، لكن لا تُحوّل المعنى ولا تبتّ من النص الأصلي.",
                "وجهة_نظر_الهيئة_بالتفصيل": "انقل بالتفصيل كل ما ورد في نص القرار عن موقف الهيئة بهذا البند: التبريرات، الأدلة أو النتائج الرقابية، طريقة تفسير الهيئة للوقائع والعقود والسجلات، وأي حجج فنية أو إجرائية ذكرتها الهيئة.",
                "رأي_اللجنة_الابتدائية_ومبرراته": "انقل نص قرار اللجنة الابتدائية في هذا البند مع الأسباب والمبررات التي استندت إليها اللجنة. اذكر تفاصيل القرار ونتيجته كما وردت نصاً.وفي حاله عدم توفرها بشكل صريح قم باستنتاجها بناء علي المعطيات",
                "رأي_اللجنة_النهائية_ومبرراته": "انقل نص قرار اللجنة النهائية في هذا البند مع الأسباب والمبررات التي استندت إليها اللجنة. لا تختصر، وأعد الأسباب كاملة أو بنحو قريب جداً مع الحفاظ على المعنى والتفصيل."
            }
            ],
            "خلاصة_نهائية": "جملة أو فقرة واحدة تُلخّص نتيجة القرار الكلية كما وردت (مثل: قبول الاستئناف شكلاً وإعادة الدعوى إلى دائرة الفصل للنظر في بنود محددة)، لكن بدون ذكر مواد قانونية أو مبالغ."
        }
        }

        قواعد تنفيذية إضافية:
        1. لا تُدرج في أي حقل عناوين مواد قانونية أو أرقام مواد أو نصوص نظامية.
        2. في حالة عدم ذكر قرار اللجنة الابتدائية قم باستنتاجه بناء على السياق.
        3. لا تذكر مبالغ مالية، ولا قيّم الخسائر أو التزامات نقدية.
        4. عند نقل وجهات النظر والأسباب، أذكر بوضوح ما استندت إليه كل جهة (مثلاً: "استندت الشركة إلى القوائم المالية المدققة والعقود المبرمة" أو "استندت الهيئة إلى نتائج الفحص المستندي") — لكن هذا فقط إذا ورد صراحة في النص.
        5. حافظ على الترتيب كما في القالب؛ لا تضف حقولًا جديدة ولا تزيل الحقول المطلوبة.
        6. أعد الناتج بصيغة JSON صالحة فقط ولا تضف أي تعليقات خارجها.
        7. قم بسرد تفصيل لقرار (اللجنة الابتدائية) في البداية وأذكر اسم اللجنة من واقع القرار.
        8. قم بسرد تفصيل لقرار (اللجنة النهائية) في البداية وأذكر اسم اللجنة من واقع القرار.
        9. لكل بند من البنود محل الدعوى، يجب أن يحتوي على (رأي ابتدائي) و(رأي نهائي) مستقلين مع المبررات كما وردت في النص.
        10. إذا كان النص المستخرج غير واضح أو به التباس، انسخه كما هو دون تعديل أو تفسير لتجنّب أي سوء فهم.
        """

    return ocr_with_gemini_retry(image_paths, instruction)

def clean_gemini_output(raw_text):
    """Cleans the raw text from Gemini to extract the JSON part."""
    if not raw_text:
        return "{}"
    
    # Find the start and end of the JSON block
    start_index = raw_text.find('{')
    end_index = raw_text.rfind('}')
    
    if start_index != -1 and end_index != -1:
        json_str = raw_text[start_index:end_index+1]
        return json_str
    
    # Return empty JSON if no valid block is found
    return "{}"

def save_progress(results, output_file):
    """Save current progress to avoid losing data."""
    try:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"  ✅ Progress saved to {output_file}")
    except Exception as e:
        print(f"  [ERROR] Could not save progress: {e}")

def main():
    """
    Main function to orchestrate the PDF processing workflow.
    Saves all extracted data into a JSON file with progress saving.
    """
    # --- CONFIGURATION ---
    pdf_input_folder = r"C:\Users\AReda\Downloads\Exise tax\Decesions\Done"
    temp_image_folder = r"C:\Users\AReda\Downloads\Exise tax\Decesions\temp_images"
    output_json_file = r"C:\Users\AReda\Downloads\Exise tax\Decesions\decisions.json"
    progress_file = r"C:\Users\AReda\Downloads\Exise tax\Decesions\progress_backup.json"
    # --- END CONFIGURATION ---

    results = []  # list of extracted dicts
    processed_files = set()  # Track processed files

    # Load existing progress if available
    if os.path.exists(progress_file):
        try:
            with open(progress_file, "r", encoding="utf-8") as f:
                results = json.load(f)
                processed_files = {item.get('Source_Filename', '') for item in results}
            print(f"Loaded {len(results)} previously processed files from progress backup.")
        except Exception as e:
            print(f"Could not load progress file: {e}")

    # Get PDF files
    pdf_files = [f for f in os.listdir(pdf_input_folder) if f.lower().endswith('.pdf')]
    remaining_files = [f for f in pdf_files if f not in processed_files]
    
    print(f"\nFound {len(pdf_files)} total PDF files.")
    print(f"Already processed: {len(processed_files)}")
    print(f"Remaining to process: {len(remaining_files)}")

    for i, filename in enumerate(remaining_files, 1):
        print(f"\n--- Processing '{filename}' ({i}/{len(remaining_files)}) ---")
        pdf_path = os.path.join(pdf_input_folder, filename)
        image_paths = []

        try:
            # 1. Convert PDF to images
            print("  Step 1: Converting PDF to images...")
            image_paths = convert_pdf_to_images(pdf_path, temp_image_folder)

            if not image_paths:
                print(f"  [ERROR] No images were created for '{filename}'. Skipping.")
                continue

            # 2. Extract data with Gemini (with retry logic)
            print("  Step 2: Extracting text with Gemini AI...")
            raw_gemini_text = ocr_complex_document(image_paths)

            if raw_gemini_text is None:
                print(f"  [ERROR] Failed to extract text for '{filename}'. Skipping.")
                continue

            # 3. Parse JSON output
            print("  Step 3: Parsing AI output...")
            json_text = clean_gemini_output(raw_gemini_text)
            
            try:
                extracted_data = json.loads(json_text)
            except json.JSONDecodeError as e:
                print(f"  [ERROR] Invalid JSON output for '{filename}': {e}")
                print(f"  Raw output: {json_text[:200]}...")
                continue

            # Add source filename
            extracted_data['Source_Filename'] = filename

            # 4. Append to results list
            results.append(extracted_data)

            print(f"  ✅ Successfully processed '{filename}'")
            
            # Save progress every 5 files
            if len(results) % 5 == 0:
                save_progress(results, progress_file)

        except Exception as e:
            print(f"  [!!!] Unexpected error while processing '{filename}': {e}")
            continue

        finally:
            # Cleanup temp images more safely
            if image_paths:
                print("  Cleaning up temporary files...")
                safe_cleanup_images(image_paths)
                time.sleep(1)  # Give system time to release file handles
            
            # Clean up temp folder if it exists
            safe_cleanup_folder(temp_image_folder)

        # Add a small delay between files to avoid rate limiting
        if i < len(remaining_files):
            print("  Waiting before next file...")
            time.sleep(2)

    print("\n--- All PDF files have been processed. ---")

    # Save final results
    try:
        with open(output_json_file, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"✅ Saved extracted data to {output_json_file}")
        
        # Clean up progress file
        if os.path.exists(progress_file):
            os.remove(progress_file)
            
    except Exception as e:
        print(f"[ERROR] Could not save final JSON file: {e}")

    return results


# Example usage
if __name__ == "__main__":
    extracted_chunks = main()
    print(f"\nProcessing complete! Total extracted documents: {len(extracted_chunks)}")
    if extracted_chunks:
        print("\nSample Extracted Data (first item):")
        print(json.dumps(extracted_chunks[0], ensure_ascii=False, indent=2))

c:\Users\AReda\anaconda3\envs\finbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Found 254 total PDF files.
Already processed: 0
Remaining to process: 254

--- Processing '1.pdf' (1/254) ---
  Step 1: Converting PDF to images...
  Step 2: Extracting text with Gemini AI...
  Attempt 1 to process with Gemini...
  Step 3: Parsing AI output...
  ✅ Successfully processed '1.pdf'
  Cleaning up temporary files...
  Waiting before next file...

--- Processing '246574-2024-E - الدائرة الاستئنافية الأولى لمخالفات ومنازعات ضريبية القيمة المضافة والسلع الانتقائية في مدينة الرياض.pdf' (2/254) ---
  Step 1: Converting PDF to images...
  [ERROR] Failed to convert PDF '246574-2024-E - الدائرة الاستئنافية الأولى لمخالفات ومنازعات ضريبية القيمة المضافة والسلع الانتقائية في مدينة الرياض.pdf' to images: Cannot open empty file: filename='C:\\Users\\AReda\\Downloads\\Exise tax\\Decesions\\Done\\246574-2024-E - الدائرة الاستئنافية الأولى لمخالفات ومنازعات ضريبية القيمة المضافة والسلع الانتقائية في مدينة الرياض.pdf'.
  [ERROR] No images were created for '246574-2024-E - الدائرة الاستئناف

# pdf to markdown

In [3]:
import os
from pathlib import Path
import PyPDF2
from markdown import markdown


def pdf_to_markdown(pdf_path, output_path=None):
    """
    Convert a single PDF file to Markdown format.
    
    Args:
        pdf_path: Path to the PDF file
        output_path: Optional output path for the .md file
    
    Returns:
        Path to the created markdown file
    """
    try:
        # Open and read the PDF
        with open(pdf_path, 'rb') as pdf_file:
            pdf_reader = PyPDF2.PdfReader(pdf_file)
            
            # Extract text from all pages
            text_content = []
            for page_num, page in enumerate(pdf_reader.pages, 1):
                text = page.extract_text()
                if text.strip():
                    text_content.append(f"## Page {page_num}\n\n{text}\n")
            
            # Combine all text
            markdown_content = "\n".join(text_content)
        
        # Determine output path
        if output_path is None:
            output_path = pdf_path.replace('.pdf', '.md')
        
        # Write to markdown file
        with open(output_path, 'w', encoding='utf-8') as md_file:
            md_file.write(markdown_content)
        
        return output_path
    
    except Exception as e:
        print(f"Error converting {pdf_path}: {str(e)}")
        return None


def convert_pdfs_folder(folder_path, output_folder=None):
    """
    Convert all PDF files in a folder to Markdown format.
    
    Args:
        folder_path: Path to the folder containing PDF files
        output_folder: Optional separate folder for output files
    
    Returns:
        List of successfully converted file paths
    """
    folder = Path(folder_path)
    
    if not folder.exists():
        print(f"Error: Folder '{folder_path}' does not exist!")
        return []
    
    # Create output folder if specified
    if output_folder:
        output_path = Path(output_folder)
        output_path.mkdir(parents=True, exist_ok=True)
    else:
        output_path = folder
    
    # Find all PDF files
    pdf_files = list(folder.glob('*.pdf'))
    
    if not pdf_files:
        print(f"No PDF files found in '{folder_path}'")
        return []
    
    print(f"Found {len(pdf_files)} PDF file(s) to convert...")
    
    converted_files = []
    
    for pdf_file in pdf_files:
        print(f"Converting: {pdf_file.name}...", end=' ')
        
        # Create output path
        output_md = output_path / pdf_file.name.replace('.pdf', '.md')
        
        # Convert the PDF
        result = pdf_to_markdown(str(pdf_file), str(output_md))
        
        if result:
            converted_files.append(result)
            print("✓ Done")
        else:
            print("✗ Failed")
    
    print(f"\nSuccessfully converted {len(converted_files)}/{len(pdf_files)} files")
    
    return converted_files


# Example usage
if __name__ == "__main__":
    # Convert all PDFs in the current directory
    # convert_pdfs_folder("./pdfs")
    
    # Or specify an output folder
    convert_pdfs_folder(r"C:\Users\AReda\Downloads\Exise tax\Laws", output_folder=r"C:\Users\AReda\Downloads\Exise tax\Laws\Laws_md")
    
    # Or convert a single PDF
    # pdf_to_markdown("example.pdf", "example.md")

Found 7 PDF file(s) to convert...
Converting: Excise Goods Tax Law.pdf... ✓ Done
Converting: ExsiseRules.pdf... ✓ Done
Converting: GCC_Unilateral_Agreement_for_Excise_Tax_Arabic.pdf... ✓ Done
Converting: Tax_Stamps_Regulations.pdf... ✓ Done
Converting: اللائحة التنفيذية للضريبة الانتقائية.pdf... ✓ Done
Converting: دليل تصنيف العقوبات.pdf... ✓ Done
Converting: ضريبة المشرومات المحلاة.pdf... ✓ Done

Successfully converted 7/7 files


# markdown to json

In [4]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
import json
import time
from pathlib import Path

# Load environment variables
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

# Configure the Gemini API
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found. Please set it in your .env file.")

genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-pro')

def read_markdown_file(file_path):
    """Reads a markdown file and returns its content."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        return content
    except Exception as e:
        print(f" [ERROR] Failed to read file '{os.path.basename(file_path)}': {e}")
        return None

def process_with_gemini_retry(markdown_content, filename, max_retries=3, base_delay=5):
    """Processes markdown content with Gemini AI with retry logic."""
    
    prompt = f"""Role: You are an expert Data Engineer and Text Editor specializing in reconstructing messy OCR text into clean, structured, and grammatically correct documents.

Task: You will receive raw text from a markdown document. Your goal is to:
* Correct any OCR errors (broken words, typos, disjointed or repeated characters).
* Reconstruct broken sentences and paragraphs.
* Remove headers, footers, page numbers, and irrelevant characters.
* Output the text in a structured JSON array according to the rules below.

Output Requirements:
* Output a JSON array, where each object represents a logical section (chunk) of the document.
* JSON Structure:
[
  {{
    "document_title": "The official title of the document",
    "chunk_title": "Title of the Chapter/Section or logical grouping",
    "content": "Full, cleaned, reconstructed text for this section with proper formatting and line breaks."
  }}
]

Processing Rules:
1. Text Cleaning:
   * Fix all OCR errors, join broken words, remove extra spaces or strange characters.
   * Ensure the text is grammatically correct and coherent.
2. Grouping:
   * Do not create a chunk for each line.
   * Group content logically by Chapters, Sections, or other natural divisions.
   * For large sections, group multiple paragraphs or items together per chunk to keep context intact.
3. Titles:
   * `document_title`: The official title of the document.
   * `chunk_title`: Describe the content, e.g., "Chapter 2: Supplies - Articles 5 to 9".
   * `content`: Include the full, cleaned text with proper formatting and line breaks.
4. Special Sections:
   * The first chunk should be the Introduction or Preamble, if present.
   * The last chunk should contain Conclusion, Signatures, or Closing Remarks, if present.

IMPORTANT: Return ONLY valid JSON array. Do not include any markdown formatting, code blocks, or explanatory text.

Input Text:
{markdown_content}
"""
    
    for attempt in range(max_retries):
        try:
            print(f"  Attempt {attempt + 1} to process with Gemini...")
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            error_msg = str(e).lower()
            if "504" in error_msg or "deadline" in error_msg or "timeout" in error_msg:
                if attempt < max_retries - 1:
                    delay = base_delay * (2 ** attempt)
                    print(f"  [WARNING] Timeout error (attempt {attempt + 1}). Retrying in {delay} seconds...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"  [ERROR] Max retries reached for timeout error: {e}")
                    return None
            elif "quota" in error_msg or "rate" in error_msg:
                if attempt < max_retries - 1:
                    delay = 60
                    print(f"  [WARNING] Rate limit/quota error. Waiting {delay} seconds...")
                    time.sleep(delay)
                    continue
                else:
                    print(f"  [ERROR] Max retries reached for rate limit: {e}")
                    return None
            else:
                print(f"  [ERROR] Unexpected error: {e}")
                return None
    
    return None

def clean_gemini_output(raw_text):
    """Cleans the raw text from Gemini to extract the JSON part."""
    if not raw_text:
        return "[]"
    
    # Remove markdown code blocks if present
    raw_text = raw_text.strip()
    if raw_text.startswith("```json"):
        raw_text = raw_text[7:]
    elif raw_text.startswith("```"):
        raw_text = raw_text[3:]
    
    if raw_text.endswith("```"):
        raw_text = raw_text[:-3]
    
    raw_text = raw_text.strip()
    
    # Find the start and end of the JSON array
    start_index = raw_text.find('[')
    end_index = raw_text.rfind(']')
    
    if start_index != -1 and end_index != -1:
        json_str = raw_text[start_index:end_index+1]
        return json_str
    
    return "[]"

def save_progress(processed_files, progress_file):
    """Save list of processed files."""
    try:
        with open(progress_file, "w", encoding="utf-8") as f:
            json.dump(list(processed_files), f, ensure_ascii=False, indent=2)
        print(f"  ✅ Progress saved")
    except Exception as e:
        print(f"  [ERROR] Could not save progress: {e}")

def main():
    """Main function to process markdown files."""
    
    # --- CONFIGURATION ---
    md_input_folder = r"C:\Users\AReda\Downloads\Exise tax\Laws\Laws_md"
    output_folder = r"C:\Users\AReda\Downloads\Exise tax\Laws\Las_json"
    progress_file = r"C:\Users\AReda\Desktop\finbot pro\progress_md_to_json.json"
    # --- END CONFIGURATION ---
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    processed_files = set()
    
    # Load existing progress if available
    if os.path.exists(progress_file):
        try:
            with open(progress_file, "r", encoding="utf-8") as f:
                processed_files = set(json.load(f))
            print(f"Loaded {len(processed_files)} previously processed files.")
        except Exception as e:
            print(f"Could not load progress file: {e}")
    
    # Get markdown files
    md_files = [f for f in os.listdir(md_input_folder) if f.lower().endswith('.md')]
    remaining_files = [f for f in md_files if f not in processed_files]
    
    print(f"\nFound {len(md_files)} total markdown files.")
    print(f"Already processed: {len(processed_files)}")
    print(f"Remaining to process: {len(remaining_files)}")
    
    for i, filename in enumerate(remaining_files, 1):
        print(f"\n--- Processing '{filename}' ({i}/{len(remaining_files)}) ---")
        md_path = os.path.join(md_input_folder, filename)
        
        try:
            # 1. Read markdown file
            print("  Step 1: Reading markdown file...")
            md_content = read_markdown_file(md_path)
            
            if not md_content:
                print(f"  [ERROR] Could not read '{filename}'. Skipping.")
                continue
            
            # 2. Process with Gemini
            print("  Step 2: Processing with Gemini AI...")
            raw_gemini_text = process_with_gemini_retry(md_content, filename)
            
            if raw_gemini_text is None:
                print(f"  [ERROR] Failed to process '{filename}'. Skipping.")
                continue
            
            # 3. Parse JSON output
            print("  Step 3: Parsing AI output...")
            json_text = clean_gemini_output(raw_gemini_text)
            
            try:
                extracted_data = json.loads(json_text)
            except json.JSONDecodeError as e:
                print(f"  [ERROR] Invalid JSON output for '{filename}': {e}")
                print(f"  Raw output: {json_text[:200]}...")
                continue
            
            # 4. Save to individual JSON file
            output_filename = os.path.splitext(filename)[0] + '.json'
            output_path = os.path.join(output_folder, output_filename)
            
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(extracted_data, f, ensure_ascii=False, indent=2)
            
            print(f"  ✅ Successfully processed '{filename}' -> '{output_filename}'")
            
            # Mark as processed
            processed_files.add(filename)
            
            # Save progress every 5 files
            if len(processed_files) % 5 == 0:
                save_progress(processed_files, progress_file)
            
        except Exception as e:
            print(f"  [!!!] Unexpected error while processing '{filename}': {e}")
            continue
        
        # Delay between files to avoid rate limiting
        if i < len(remaining_files):
            print("  Waiting before next file...")
            time.sleep(2)
    
    print("\n--- All markdown files have been processed. ---")
    
    # Save final progress
    save_progress(processed_files, progress_file)
    
    # Clean up progress file
    if os.path.exists(progress_file) and len(processed_files) == len(md_files):
        os.remove(progress_file)
        print("✅ All files processed. Progress file removed.")
    
    print(f"\n✅ Processing complete! Total processed files: {len(processed_files)}")
    print(f"📁 JSON files saved to: {output_folder}")

if __name__ == "__main__":
    main()

Loaded 110 previously processed files.

Found 7 total markdown files.
Already processed: 110
Remaining to process: 7

--- Processing 'Excise Goods Tax Law.md' (1/7) ---
  Step 1: Reading markdown file...
  Step 2: Processing with Gemini AI...
  Attempt 1 to process with Gemini...
  Step 3: Parsing AI output...
  ✅ Successfully processed 'Excise Goods Tax Law.md' -> 'Excise Goods Tax Law.json'
  Waiting before next file...

--- Processing 'ExsiseRules.md' (2/7) ---
  Step 1: Reading markdown file...
  Step 2: Processing with Gemini AI...
  Attempt 1 to process with Gemini...
  Step 3: Parsing AI output...
  ✅ Successfully processed 'ExsiseRules.md' -> 'ExsiseRules.json'
  Waiting before next file...

--- Processing 'GCC_Unilateral_Agreement_for_Excise_Tax_Arabic.md' (3/7) ---
  Step 1: Reading markdown file...
  Step 2: Processing with Gemini AI...
  Attempt 1 to process with Gemini...
  Step 3: Parsing AI output...
  ✅ Successfully processed 'GCC_Unilateral_Agreement_for_Excise_Tax_Ara

In [ ]:
# import google.generativeai as genai
# import json
# import os

# # Configure Gemini
# genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# def json_to_sample_text(json_data: dict) -> str:
#     """
#     Convert a JSON document with title, paragraph, and chunk_content 
#     into plain sample text using Gemini 2.5-flash.
#     """
#     model = genai.GenerativeModel("gemini-2.5-flash")

#     prompt = """
# You are given a JSON object with the following fields:
# - "title": document title
# - "paragraph": introduction text in HTML
# - "chunk_title": section title
# - "chunk_content": section content in HTML

# Your task:
# 1. Convert the HTML fields ("paragraph" and "chunk_content") into clean plain text.
# 2. Rewrite the output as a structured sample text in Arabic that includes:
#    - The title
#    - The introduction (paragraphs)
#    - The chunk title
#    - The chunk content
# 3. Do not include any HTML tags, only plain readable text don't include any examples mentioned in the chunk, if the chunk is too big and have a lot of numbers summarize it.
# 4. Preserve Arabic meaning and readability.
# """

#     response = model.generate_content([prompt, json.dumps(json_data, ensure_ascii=False)])
#     return response.text



# # Example usage
# if __name__ == "__main__":
#     # read a sample JSON file
#     sample_json_path = r"C:\Users\AReda\Desktop\finbot pro\html_chunks.json"
#     # loop through the JSON file and process each entry
#     with open(sample_json_path, "r", encoding="utf-8") as f:
#         sample_json_list = json.load(f)
#     for sample_json in sample_json_list:
#         sample_text = json_to_sample_text(sample_json)
#         # append the result to a json file
#         output_text_file = r"C:\Users\AReda\Desktop\finbot pro\sample_text.txt"
#         with open(output_text_file, "a", encoding="utf-8") as f:
#             f.write(sample_text + "\n\n---\n\n")

In [ ]:
# import requests
# from bs4 import BeautifulSoup
# import json
# import time
# import csv

# BASE_URL = "https://www.vezeeta.com/en/doctor/dentistry/egypt?page={}"
# HEADERS = {"User-Agent": "Mozilla/5.0"}

# all_doctors = []

# for page in range(1, 20):  # adjust the range (20 pages = ~200 doctors)
#     print(f"Scraping page {page}...")
#     url = BASE_URL.format(page)
#     response = requests.get(url, headers=HEADERS)

#     if response.status_code != 200:
#         print("Failed to fetch page", page)
#         break

#     soup = BeautifulSoup(response.text, "html.parser")

#     doctors_found = 0
#     for script in soup.find_all("script", {"type": "application/ld+json"}):
#         try:
#             data = json.loads(script.string)
#             if "name" in data and "@type" in data:
#                 all_doctors.append({
#                     "name": data.get("name"),
#                     "specialty": data.get("medicalSpecialty"),
#                     "description": data.get("description"),
#                     "address": data.get("address", {}).get("streetAddress"),
#                     "area": data.get("areaServed"),
#                     "rating": data.get("aggregateRating", {}).get("ratingValue"),
#                     "reviews": data.get("aggregateRating", {}).get("ratingCount"),
#                     "fees": data.get("priceRange"),
#                     "phone": data.get("telephone"),
#                     "url": "https://www.vezeeta.com" + data.get("url", ""),
#                     "image": data.get("image")
#                 })
#                 doctors_found += 1
#         except Exception:
#             pass

#     if doctors_found == 0:  # stop if no new doctors
#         print("No doctors found on page", page, "— stopping.")
#         break

#     time.sleep(2)  # avoid getting blocked

# # Save to CSV
# with open("doctors_data.csv", "w", newline="", encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=all_doctors[0].keys())
#     writer.writeheader()
#     writer.writerows(all_doctors)

# print(f"✅ Scraped {len(all_doctors)} doctors in total.")

# import os
# import shutil
# import fitz  # PyMuPDF
# from PIL import Image
# from dotenv import load_dotenv
# import google.generativeai as genai
# import time
# import gc
# print('FinBot/gmini_code_t.py loaded')

# # --- API Setup ---
# load_dotenv()
# api_key = os.getenv('GOOGLE_API_KEY')
# if not api_key:
#     raise ValueError("GOOGLE_API_KEY not found. Please set it in your .env file.")
# genai.configure(api_key=api_key)

# # Gemini Model
# model = genai.GenerativeModel('gemini-2.5-flash')


# def convert_pdf_to_images(pdf_path, output_folder, dpi=200):
#     """Convert each page of a PDF to PNG images."""
#     os.makedirs(output_folder, exist_ok=True)
#     image_paths = []
#     pdf_document = None
#     try:
#         pdf_document = fitz.open(pdf_path)
#         for page_number in range(pdf_document.page_count):
#             page = pdf_document[page_number]
#             pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
#             output_path = os.path.join(output_folder, f"page_{page_number + 1}.png")
#             pix.save(output_path)
#             image_paths.append(output_path)
#             # Clean up pixmap
#             pix = None
#     except Exception as e:
#         print(f"[ERROR] Failed to convert PDF '{os.path.basename(pdf_path)}' to images: {e}")
#         return []
#     finally:
#         # Properly close the PDF document
#         if pdf_document:
#             pdf_document.close()
#     return image_paths


# def ocr_with_gemini(image_paths, instruction):
#     """Extract text from images using Gemini OCR."""
#     images = []
#     try:
#         # Open images
#         for path in image_paths:
#             img = Image.open(path)
#             # Convert to RGB if needed to avoid issues
#             if img.mode != 'RGB':
#                 img = img.convert('RGB')
#             images.append(img)
        
#         prompt = f"""
#         {instruction}

#         These are pages from a PDF. Extract all text content while preserving structure.
#         Pay attention to tables, columns, headers, and formatting.
#         """
#         response = model.generate_content([prompt, *images])
#         return response.text
#     finally:
#         # Explicitly close all image objects
#         for img in images:
#             try:
#                 img.close()
#             except:
#                 pass
#         # Force garbage collection
#         gc.collect()


# def ocr_complex_document(image_paths):
#     """Use Gemini to extract full text as HTML."""
#     return ocr_with_gemini(image_paths, "Extract the whole text from the document in HTML format while preserving structure and layout and formatting." \
#     " Ensure tables and lists are properly formatted in HTML." \
#     " Use appropriate HTML tags like <h1>, <h2>, <p>, <table>, <ul>, <li>, etc." \
#     " Maintain paragraph breaks and spacing. No need to add styling or colors." \
#     " no need for intro, المحتويات او الخاتمة, just the main content." \
#     "Don't forget we want all text in the pdf to be extracted.")


# def safe_remove_directory(directory_path, max_attempts=5, delay=1):
#     """Safely remove directory with retry mechanism."""
#     for attempt in range(max_attempts):
#         try:
#             if os.path.exists(directory_path):
#                 # Force garbage collection before attempting removal
#                 gc.collect()
#                 time.sleep(delay)
#                 shutil.rmtree(directory_path)
#             return True
#         except PermissionError as e:
#             print(f"  [WARNING] Attempt {attempt + 1}/{max_attempts} failed to remove temp folder: {e}")
#             if attempt < max_attempts - 1:
#                 time.sleep(delay * (attempt + 1))  # Increasing delay
#             else:
#                 print(f"  [ERROR] Failed to remove temp folder after {max_attempts} attempts")
#                 return False
#         except Exception as e:
#             print(f"  [ERROR] Unexpected error removing temp folder: {e}")
#             return False
#     return False


# def main():
#     pdf_input_folder = r"C:\Users\AReda\Desktop\finbot pro\data\guides"
#     output_folder = r"C:\Users\AReda\Desktop\finbot pro\html_outputs"
#     temp_image_folder = r"C:\Users\AReda\Desktop\finbot pro\temp_images"

#     os.makedirs(output_folder, exist_ok=True)

#     pdf_files = [f for f in os.listdir(pdf_input_folder) if f.lower().endswith('.pdf')]
#     print(f"\nFound {len(pdf_files)} PDF files to process.")

#     for filename in pdf_files:
#         print(f"\n--- Processing '{filename}' ---")
#         pdf_path = os.path.join(pdf_input_folder, filename)

#         try:
#             # Convert PDF → Images
#             print("  Step 1: Converting PDF to images...")
#             image_paths = convert_pdf_to_images(pdf_path, temp_image_folder)
#             if not image_paths:
#                 print(f"[ERROR] No images created for '{filename}'. Skipping.")
#                 continue

#             # OCR with Gemini
#             print("  Step 2: Extracting text with Gemini AI...")
#             html_text = ocr_complex_document(image_paths)

#             # Save as HTML
#             output_html_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.html")
#             with open(output_html_path, "w", encoding="utf-8") as f:
#                 f.write(html_text)

#             print(f"  ✅ Saved HTML to: {output_html_path}")

#         except Exception as e:
#             print(f"[!!!] Error while processing '{filename}': {e}")
#         finally:
#             # Clean up temp images with retry mechanism
#             print("  Step 3: Cleaning up temporary files...")
#             if not safe_remove_directory(temp_image_folder):
#                 print(f"  [WARNING] Could not remove temp folder. You may need to delete it manually: {temp_image_folder}")

#     print("\n--- All PDF files processed. ---")


# if __name__ == "__main__":
#     main()
# import requests
# from bs4 import BeautifulSoup
# import json

# url = "https://www.vezeeta.com/en/doctor/dentistry/egypt"  # Example: Dentists in Egypt
# headers = {"User-Agent": "Mozilla/5.0"}

# response = requests.get(url, headers=headers)
# soup = BeautifulSoup(response.text, "html.parser")

# doctors = []
# for script in soup.find_all("script", {"type": "application/ld+json"}):
#     try:
#         data = json.loads(script.string)
#         if "name" in data and "@type" in data:
#             doctors.append({
#                 "name": data.get("name"),
#                 "specialty": data.get("medicalSpecialty"),
#                 "description": data.get("description"),
#                 "address": data.get("address", {}).get("streetAddress"),
#                 "area": data.get("areaServed"),
#                 "rating": data.get("aggregateRating", {}).get("ratingValue"),
#                 "reviews": data.get("aggregateRating", {}).get("ratingCount"),
#                 "fees": data.get("priceRange"),
#                 "phone": data.get("telephone"),
#                 "url": "https://www.vezeeta.com" + data.get("url", ""),
#                 "image": data.get("image")
#             })
#     except Exception:
#         pass

# print(doctors[:100])  # preview first 3 doctors
# import os
# import json
# from bs4 import BeautifulSoup, NavigableString

# def chunk_html_with_repeated_header(html_content):
#     """
#     Chunks an HTML string into a list of JSON objects based on h2 tags.
#     Each JSON object will contain:
#     - the main document title (h1),
#     - the introductory paragraph,
#     - the chunk title (h2),
#     - the chunk content.
#     """
#     soup = BeautifulSoup(html_content, 'html.parser')
#     chunks = []

#     # --- 1. Extract the main title and intro paragraphs ---
#     main_title_tag = soup.find('h1')
#     main_title = main_title_tag.get_text(strip=True) if main_title_tag else ""
    
#     intro_paragraphs = []
#     if main_title_tag:
#         for sibling in main_title_tag.find_next_siblings():
#             if sibling.name == 'h2':
#                 break
#             if sibling.name and sibling.get_text(strip=True):
#                 intro_paragraphs.append(str(sibling))
#     intro_paragraph_content = "\n".join(intro_paragraphs)

#     # --- 2. Extract h2-based chunks ---
#     h2_tags = soup.find_all('h2')
    
#     if not h2_tags:
#         body_content = soup.body.decode_contents() if soup.body else ""
#         chunks.append({
#             "title": main_title,
#             "paragraph": intro_paragraph_content.strip(),
#             "chunk_title": main_title,
#             "chunk_content": body_content.strip()
#         })
#     else:
#         for h2 in h2_tags:
#             chunk_title = h2.get_text(strip=True)
#             content_elements = []
#             for sibling in h2.find_next_siblings():
#                 if sibling.name == 'h2':
#                     break
#                 if not (isinstance(sibling, NavigableString) and not sibling.strip()):
#                     content_elements.append(str(sibling))
            
#             chunk_content = "\n".join(content_elements)
#             chunks.append({
#                 "title": main_title,
#                 "paragraph": intro_paragraph_content.strip(),
#                 "chunk_title": chunk_title,
#                 "chunk_content": chunk_content.strip()
#             })
        
#     return chunks


# # --- Loop through all HTML files in a folder ---
# input_folder = r"C:\Users\AReda\Desktop\finbot pro\html_outputs"
# output_json_file = r"C:\Users\AReda\Desktop\finbot pro\html_chunk.json"

# all_chunks = []

# for file_name in os.listdir(input_folder):
#     if file_name.lower().endswith(".html"):
#         file_path = os.path.join(input_folder, file_name)
#         try:
#             with open(file_path, "r", encoding="utf-8") as f:
#                 html_data = f.read()
            
#             chunks = chunk_html_with_repeated_header(html_data)
#             all_chunks.extend(chunks)  # merge all chunks
#             print(f"✅ Processed {file_name}, extracted {len(chunks)} chunks")
#         except Exception as e:
#             print(f"[ERROR] Failed to process {file_name}: {e}")

# # --- Save all chunks into one JSON file ---
# try:
#     # Load existing data if file exists
#     if os.path.exists(output_json_file):
#         with open(output_json_file, "r", encoding="utf-8") as f:
#             existing_data = json.load(f)
#     else:
#         existing_data = []

#     # Append new chunks
#     existing_data.extend(all_chunks)

#     with open(output_json_file, "w", encoding="utf-8") as f:
#         json.dump(existing_data, f, ensure_ascii=False, indent=2)
#     print(f"\n📂 All chunks saved to {output_json_file}")
# except Exception as e:
#     print(f"[ERROR] Could not save JSON file: {e}")
# import json
# from bs4 import BeautifulSoup, NavigableString

# def chunk_html_with_repeated_header(html_content):
#     """
#     Chunks an HTML string into a list of JSON objects based on h2 tags.

#     Each JSON object in the list will contain the main document title (h1),
#     the introductory paragraph, the title of the specific chunk (h2), and
#     the content of that chunk.
#     """
#     soup = BeautifulSoup(html_content, 'html.parser')
#     chunks = []

#     # --- 1. Extract the main title and introductory paragraph to be used in all chunks ---
#     main_title_tag = soup.find('h1')
#     main_title = main_title_tag.get_text(strip=True) if main_title_tag else ""
    
#     intro_paragraphs = []
#     if main_title_tag:
#         for sibling in main_title_tag.find_next_siblings():
#             # Stop collecting paragraphs when the first h2 is reached
#             if sibling.name == 'h2':
#                 break
#             # Only include non-empty tags
#             if sibling.name and sibling.get_text(strip=True):
#                  intro_paragraphs.append(str(sibling))

#     intro_paragraph_content = "\n".join(intro_paragraphs)

#     # --- 2. Find all h2 tags to define the start of each chunk ---
#     h2_tags = soup.find_all('h2')
    
#     if not h2_tags:
#         # If there are no h2 tags, create a single chunk with the available content
#         body_content = soup.body.decode_contents() if soup.body else ""
#         chunk_data = {
#             "title": main_title,
#             "paragraph": intro_paragraph_content.strip(),
#             "chunk_title": main_title, # Fallback to main title if no h2
#             "chunk_content": body_content
#         }
#         chunks.append(chunk_data)
#     else:
#         # --- 3. Iterate through each h2 section and create a chunk ---
#         for h2 in h2_tags:
#             chunk_title = h2.get_text(strip=True)
            
#             # Collect all sibling elements until the next h2 tag
#             content_elements = []
#             for sibling in h2.find_next_siblings():
#                 if sibling.name == 'h2':
#                     break
#                 # Check if the sibling is not just a whitespace NavigableString
#                 if not (isinstance(sibling, NavigableString) and not sibling.strip()):
#                     content_elements.append(str(sibling))
            
#             chunk_content = "\n".join(content_elements)
            
#             # Create the JSON object for the chunk
#             chunk_data = {
#                 "title": main_title,
#                 "paragraph": intro_paragraph_content.strip(),
#                 "chunk_title": chunk_title,
#                 "chunk_content": chunk_content.strip()
#             }
#             chunks.append(chunk_data)
        
#     return json.dumps(chunks, indent=4, ensure_ascii=False)

# with open(r'C:\Users\AReda\Desktop\finbot pro\html_outputs\دليل معالجة ديون مكلفي الزكاة - الإصدار الثاني.html', "r", encoding="utf-8") as f:
#     html_data = f.read()
# intro_section = chunk_html_with_repeated_header(html_data)

# # Print the result as a formatted JSON object
# print(intro_section)

# # Append the result to a JSON file
# output_json_file = r'C:\Users\AReda\Desktop\finbot pro\html_chunks.json'
# try:
#     # Load existing data if file exists
#     if os.path.exists(output_json_file):
#         with open(output_json_file, "r", encoding="utf-8") as f:
#             existing_data = json.load(f)
#     else:
#         existing_data = []

#     # Append new chunk(s)
#     existing_data.append(json.loads(intro_section))

#     # Save back to file
#     with open(output_json_file, "w", encoding="utf-8") as f:
#         json.dump(existing_data, f, ensure_ascii=False, indent=2)
#     print(f"✅ Appended to {output_json_file}")
# except Exception as e:
#     print(f"[ERROR] Could not append to JSON file: {e}")

# from bs4 import BeautifulSoup
# import json

# def chunk_full_html_document(html_content):
#     """
#     تقوم هذه الدالة بتقسيم مستند HTML بالكامل إلى قائمة منظمة من كائنات JSON.
#     الجزء الأول يحتوي على العنوان الرئيسي (h1) ومقدمته.
#     الأجزاء اللاحقة تعتمد على عناوين h2 والمحتوى الخاص بها.

#     Args:
#         html_content (str): سلسلة نصية تحتوي على مستند HTML الكامل.

#     Returns:
#         list: قائمة من الكائنات (dictionaries)، حيث يمثل كل كائن جزءًا يحتوي على "title" و "paragraph".
#     """
#     soup = BeautifulSoup(html_content, 'html.parser')
#     chunks = []

#     # --- الخطوة 1: استخراج جزء المقدمة ---
#     main_title_tag = soup.find('h1')
#     if main_title_tag:
#         title = main_title_tag.get_text(strip=True)
#         intro_paragraphs = []
        
#         # جمع المحتوى بعد H1 حتى أول H2
#         for element in main_title_tag.find_next_siblings():
#             if element.name == 'h2':
#                 break
            
#             text = element.get_text(separator='\n', strip=True)
#             if text:
#                 intro_paragraphs.append(text)
        
#         intro_text = "\n\n".join(intro_paragraphs)
        
#         chunks.append({
#             "title": title,
#             "paragraph": intro_text
#         })

#     # --- الخطوة 2: استخراج الأجزاء الخاصة بكل H2 ---
#     h2_tags = soup.find_all('h2')
#     for h2 in h2_tags:
#         section_title = h2.get_text(strip=True)
#         content_parts = []
        
#         # جمع المحتوى بعد H2 الحالي وحتى H2 التالي
#         for sibling in h2.find_next_siblings():
#             if sibling.name == 'h2':
#                 break  # التوقف عند القسم التالي
            
#             text = sibling.get_text(separator='\n', strip=True)
#             if text:
#                 content_parts.append(text)
                
#         section_paragraph = "\n\n".join(content_parts)
        
#         chunks.append({
#             "title": section_title,
#             "paragraph": section_paragraph
#         })
        
#     return chunks
# with open(r'C:\Users\AReda\Desktop\finbot pro\html_outputs\Rett&Constructions.html', "r", encoding="utf-8") as f:
#     html_data = f.read()
# intro_section = chunk_full_html_document(html_data)

# # Print the result as a formatted JSON object
# print(intro_section)

# import os
# from mistralai import Mistral

# api_key = os.environ["MISTRAL_API_KEY"]
# model = "mistral-embed"

# client = Mistral(api_key=api_key)

# embeddings_batch_response = client.embeddings.create(
#     model=model,
#     inputs=["Embed this sentence.", "As well as this one."],
# )

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...
Scraping page 6...
Scraping page 7...
Scraping page 8...
Scraping page 9...
Scraping page 10...
Scraping page 11...
Scraping page 12...
Scraping page 13...
Scraping page 14...
Scraping page 15...
Scraping page 16...
Scraping page 17...
Scraping page 18...
Scraping page 19...
✅ Scraped 190 doctors in total.


In [ ]:
# import re

# def chunk_html_by_topics(html_content):
#     """
#     تقوم هذه الدالة بتقسيم محتوى HTML إلى أجزاء بناءً على العناوين الرئيسية (h2).

#     Args:
#         html_content (str): سلسلة نصية تحتوي على كود الـ HTML الكامل.

#     Returns:
#         dict: قاموس يحتوي على عنوان رئيسي للمقدمة ومحتواها، 
#               وقائمة من القواميس لكل موضوع، حيث يحتوي كل قاموس على عنوان الموضوع ومحتواه.
#     """
    
#     # استخلاص العنوان الرئيسي للمستند من وسم <h1>
#     h1_match = re.search(r'<h1>(.*?)</h1>', html_content, re.DOTALL)
#     document_title = h1_match.group(1).strip() if h1_match else "المقدمة"

#     # تقسيم المحتوى عند كل وسم <h2> مع الاحتفاظ بالوسم
#     # نستخدم التعبير (lookahead) للإبقاء على المحدد في النتائج
#     chunks = re.split(r'(?=<h2>)', html_content)
    
#     processed_chunks = {}
    
#     # الجزء الأول هو المقدمة (قبل أول <h2>)
#     intro_content = chunks[0]
#     # إزالة وسم h1 من محتوى المقدمة لأنه استُخلص بالفعل
#     intro_content = re.sub(r'<h1>.*?</h1>', '', intro_content, flags=re.DOTALL).strip()
#     processed_chunks['introduction'] = {
#         'topic': document_title,
#         'content': intro_content
#     }
    
#     topic_chunks = []

#     # معالجة بقية الأجزاء التي تبدأ بـ <h2>
#     for i in range(1, len(chunks)):
#         chunk = chunks[i]
        
#         # استخلاص عنوان الموضوع من وسم <h2>
#         header_match = re.match(r'<h2>(.*?)</h2>', chunk, re.DOTALL)
        
#         if header_match:
#             topic_title = header_match.group(1).strip()
#             # المحتوى هو كل ما يأتي بعد وسم </h2>
#             content = chunk[header_match.end():].strip()
            
#             topic_chunks.append({
#                 'topic': topic_title,
#                 'content': content
#             })

#     processed_chunks['topics'] = topic_chunks
    
#     return processed_chunks

# print('FinBot/gmini_code_t.py loaded')
# with open(r"C:\Users\AReda\Desktop\finbot pro\html_outputs\المعالجة الزكوية.html", "r", encoding="utf-8") as f:
#     content = f.read()
# print(chunk_html_by_topics(content))
# import os
# import google.generativeai as genai
# from PIL import Image
# import base64
# import io
# import fitz  # PyMuPDF
# import os
# from dotenv import load_dotenv
# import csv
# import json
# import shutil
# import time
# import gc
# from pathlib import Path

# # ---
# # Your existing setup and functions (with improvements)
# # ---
# load_dotenv()
# api_key = os.getenv('GOOGLE_API_KEY')

# # Configure the Gemini API
# if not api_key:
#     raise ValueError("GOOGLE_API_KEY not found. Please set it in your .env file.")
# genai.configure(api_key=api_key)



# model = genai.GenerativeModel(
#     'gemini-2.5-pro',  # Changed to 1.5-pro for better stability
# )

# def convert_pdf_to_images(pdf_path, output_folder, dpi=120):  # Reduced DPI for faster processing
#     """Converts each page of a PDF to a PNG image."""
#     os.makedirs(output_folder, exist_ok=True)
#     image_paths = []
#     try:
#         pdf_document = fitz.open(pdf_path)
#         for page_number in range(len(pdf_document)):
#             page = pdf_document[page_number]
#             # Reduced resolution for faster processing and smaller file sizes
#             pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
#             output_path = os.path.join(output_folder, f"page_{page_number + 1}.png")
#             pix.save(output_path)
#             image_paths.append(output_path)
#         pdf_document.close()
#     except Exception as e:
#         print(f"  [ERROR] Failed to convert PDF '{os.path.basename(pdf_path)}' to images: {e}")
#         return []
#     return image_paths

# def safe_cleanup_images(image_paths, max_retries=3):
#     """Safely cleanup image files with retry logic."""
#     for image_path in image_paths:
#         if os.path.exists(image_path):
#             for attempt in range(max_retries):
#                 try:
#                     os.remove(image_path)
#                     break
#                 except PermissionError:
#                     print(f"  [WARNING] Could not delete {image_path} (attempt {attempt + 1}), retrying...")
#                     time.sleep(1)  # Wait a bit before retrying
#                     gc.collect()  # Force garbage collection
#                 except Exception as e:
#                     print(f"  [ERROR] Failed to delete {image_path}: {e}")
#                     break

# def safe_cleanup_folder(folder_path, max_retries=3):
#     """Safely cleanup temporary folder with retry logic."""
#     if os.path.exists(folder_path):
#         for attempt in range(max_retries):
#             try:
#                 shutil.rmtree(folder_path)
#                 break
#             except PermissionError:
#                 print(f"  [WARNING] Could not delete folder {folder_path} (attempt {attempt + 1}), retrying...")
#                 time.sleep(2)
#                 gc.collect()
#             except Exception as e:
#                 print(f"  [ERROR] Failed to delete folder {folder_path}: {e}")
#                 break

# def ocr_with_gemini_retry(image_paths, instruction, max_retries=3, base_delay=5):
#     """Processes images with Gemini OCR with retry logic for timeout errors."""
    
#     # Close PIL images properly to avoid file locks
#     images = []
#     try:
#         for path in image_paths:
#             img = Image.open(path)
#             # Convert to bytes to avoid file locks
#             img_byte_arr = io.BytesIO()
#             img.save(img_byte_arr, format='PNG')
#             img_byte_arr.seek(0)
#             images.append(Image.open(img_byte_arr))
#             img.close()  # Close the original file handle
#     except Exception as e:
#         print(f"  [ERROR] Failed to load images: {e}")
#         return None
    
#     prompt = f"""
#     {instruction}
    
#     These are pages from a PDF document. Extract all text content while preserving the structure.
#     Pay special attention to tables, columns, headers, and any structured content.
#     Maintain paragraph breaks and formatting.
#     """
    
#     for attempt in range(max_retries):
#         try:
#             print(f"  Attempt {attempt + 1} to process with Gemini...")
#             response = model.generate_content([prompt, *images])
#             return response.text
            
#         except Exception as e:
#             error_msg = str(e).lower()
#             if "504" in error_msg or "deadline" in error_msg or "timeout" in error_msg:
#                 if attempt < max_retries - 1:
#                     delay = base_delay * (2 ** attempt)  # Exponential backoff
#                     print(f"  [WARNING] Timeout error (attempt {attempt + 1}). Retrying in {delay} seconds...")
#                     time.sleep(delay)
#                     continue
#                 else:
#                     print(f"  [ERROR] Max retries reached for timeout error: {e}")
#                     return None
#             elif "quota" in error_msg or "rate" in error_msg:
#                 if attempt < max_retries - 1:
#                     delay = 60  # Wait longer for quota issues
#                     print(f"  [WARNING] Rate limit/quota error. Waiting {delay} seconds...")
#                     time.sleep(delay)
#                     continue
#                 else:
#                     print(f"  [ERROR] Max retries reached for rate limit: {e}")
#                     return None
#             else:
#                 print(f"  [ERROR] Unexpected error: {e}")
#                 return None
    
#     return None

# def ocr_complex_document(image_paths):
#     """Creates the specific prompt and calls the Gemini OCR function with retry."""
#     instruction = """
#     Extract the following fields from these document pages:

#     1. "رقم القرار" (Decision Number) — Extract exactly as written it starts with IR or VA it at the top of the 1st page
#     2. "رقم الدعوى / الاستْناف" (Appeal Number) — Extract exactly as written it starts with z or v
#     3. "العام الضريبي المرتبط بالقرار" (Tax Year related to the decision) — Extract the full year or date if present.
#     4. "اسباب القرار" (Reasons for the decision) — Extract the complete section of reasons, keeping paragraph breaks without changing the text.
#     5. "البنود محل الاعتراض" (Items under objection) — Extract the full list of items if present, preserving any numbering or bullet points if present without changing the text.
#         6. "منطوق القرار" (Decision Text) — Extract the full text of the decision, preserving formatting if present without changing the text.
#         7. "* قم بسرد تفصيل لقرار (اللجنة الابتدائية) في البداية وأذكر اسم اللجنة من واقع القرار.
#             * قم بسرد تفصيل لقرار (اللجنة النهائية) في البداية وأذكر اسم اللجنة من واقع القرار."
#     Output the result strictly in JSON format like this:
#     {
#         "رقم القرار": "...",
#         "رقم الدعوى / الاستْناف": "...",
#         "العام الضريبي المرتبط بالقرار": "...",
#         "اسباب القرار": "...",
#         "البنود محل الاعتراض": "...",
#         "منطوق القرار": "...",
#         "تفصيل القرار": {
#             "اللجنة الابتدائية": "...",
#             "اللجنة النهائية": "..."
#         }
#     }

#     Rules:
#     - Do NOT add extra commentary or explanations.
#     - Preserve the Arabic text exactly as in the document.
#     - If any field is not found, return it as an empty string.
#     """

#     return ocr_with_gemini_retry(image_paths, instruction)

# def clean_gemini_output(raw_text):
#     """Cleans the raw text from Gemini to extract the JSON part."""
#     if not raw_text:
#         return "{}"
    
#     # Find the start and end of the JSON block
#     start_index = raw_text.find('{')
#     end_index = raw_text.rfind('}')
    
#     if start_index != -1 and end_index != -1:
#         json_str = raw_text[start_index:end_index+1]
#         return json_str
    
#     # Return empty JSON if no valid block is found
#     return "{}"

# def save_progress(results, output_file):
#     """Save current progress to avoid losing data."""
#     try:
#         with open(output_file, "w", encoding="utf-8") as f:
#             json.dump(results, f, ensure_ascii=False, indent=2)
#         print(f"  ✅ Progress saved to {output_file}")
#     except Exception as e:
#         print(f"  [ERROR] Could not save progress: {e}")

# def main():
#     """
#     Main function to orchestrate the PDF processing workflow.
#     Saves all extracted data into a JSON file with progress saving.
#     """
#     # --- CONFIGURATION ---
#     pdf_input_folder = r"C:\Users\AReda\Desktop\finbot pro\data\decisions"
#     temp_image_folder = r"C:\Users\AReda\Desktop\finbot pro\temp_images"
#     output_json_file = r"C:\Users\AReda\Desktop\finbot pro\decisions.json"
#     progress_file = r"C:\Users\AReda\Desktop\finbot pro\progress_backup_new.json"
#     # --- END CONFIGURATION ---

#     results = []  # list of extracted dicts
#     processed_files = set()  # Track processed files

#     # Load existing progress if available
#     if os.path.exists(progress_file):
#         try:
#             with open(progress_file, "r", encoding="utf-8") as f:
#                 results = json.load(f)
#                 processed_files = {item.get('Source_Filename', '') for item in results}
#             print(f"Loaded {len(results)} previously processed files from progress backup.")
#         except Exception as e:
#             print(f"Could not load progress file: {e}")

#     # Get PDF files
#     pdf_files = [f for f in os.listdir(pdf_input_folder) if f.lower().endswith('.pdf')]
#     remaining_files = [f for f in pdf_files if f not in processed_files]
    
#     print(f"\nFound {len(pdf_files)} total PDF files.")
#     print(f"Already processed: {len(processed_files)}")
#     print(f"Remaining to process: {len(remaining_files)}")

#     for i, filename in enumerate(remaining_files, 1):
#         print(f"\n--- Processing '{filename}' ({i}/{len(remaining_files)}) ---")
#         pdf_path = os.path.join(pdf_input_folder, filename)
#         image_paths = []

#         try:
#             # 1. Convert PDF to images
#             print("  Step 1: Converting PDF to images...")
#             image_paths = convert_pdf_to_images(pdf_path, temp_image_folder)

#             if not image_paths:
#                 print(f"  [ERROR] No images were created for '{filename}'. Skipping.")
#                 continue

#             # 2. Extract data with Gemini (with retry logic)
#             print("  Step 2: Extracting text with Gemini AI...")
#             raw_gemini_text = ocr_complex_document(image_paths)

#             if raw_gemini_text is None:
#                 print(f"  [ERROR] Failed to extract text for '{filename}'. Skipping.")
#                 continue

#             # 3. Parse JSON output
#             print("  Step 3: Parsing AI output...")
#             json_text = clean_gemini_output(raw_gemini_text)
            
#             try:
#                 extracted_data = json.loads(json_text)
#             except json.JSONDecodeError as e:
#                 print(f"  [ERROR] Invalid JSON output for '{filename}': {e}")
#                 print(f"  Raw output: {json_text[:200]}...")
#                 continue

#             # Add source filename
#             extracted_data['Source_Filename'] = filename

#             # 4. Append to results list
#             results.append(extracted_data)

#             print(f"  ✅ Successfully processed '{filename}'")
            
#             # Save progress every 5 files
#             if len(results) % 5 == 0:
#                 save_progress(results, progress_file)

#         except Exception as e:
#             print(f"  [!!!] Unexpected error while processing '{filename}': {e}")
#             continue

#         finally:
#             # Cleanup temp images more safely
#             if image_paths:
#                 print("  Cleaning up temporary files...")
#                 safe_cleanup_images(image_paths)
#                 time.sleep(1)  # Give system time to release file handles
            
#             # Clean up temp folder if it exists
#             safe_cleanup_folder(temp_image_folder)

#         # Add a small delay between files to avoid rate limiting
#         if i < len(remaining_files):
#             print("  Waiting before next file...")
#             time.sleep(2)

#     print("\n--- All PDF files have been processed. ---")

#     # Save final results
#     try:
#         with open(output_json_file, "w", encoding="utf-8") as f:
#             json.dump(results, f, ensure_ascii=False, indent=2)
#         print(f"✅ Saved extracted data to {output_json_file}")
        
#         # Clean up progress file
#         if os.path.exists(progress_file):
#             os.remove(progress_file)
            
#     except Exception as e:
#         print(f"[ERROR] Could not save final JSON file: {e}")

#     return results


# # Example usage
# if __name__ == "__main__":
#     extracted_chunks = main()
#     print(f"\nProcessing complete! Total extracted documents: {len(extracted_chunks)}")
#     if extracted_chunks:
#         print("\nSample Extracted Data (first item):")
#         print(json.dumps(extracted_chunks[0], ensure_ascii=False, indent=2))

# import os
# import shutil
# import fitz  # PyMuPDF
# from PIL import Image
# from dotenv import load_dotenv
# import google.generativeai as genai
# import time
# from dotenv import load_dotenv
# load_dotenv()
# print('FinBot/gmini_code_t.py loaded')

# # --- API Setup ---
# load_dotenv()
# api_key = os.getenv('GOOGLE_API_KEY')
# if not api_key:
#     raise ValueError("GOOGLE_API_KEY not found. Please set it in your .env file.")
# genai.configure(api_key=api_key)

# # Gemini Model
# model = genai.GenerativeModel('gemini-2.5-flash')


# # =========================================================
# # 1. PDF → Images
# # =========================================================
# def convert_pdf_to_images(pdf_path, output_folder, dpi=200):
#     """Convert each page of a PDF to PNG images."""
#     os.makedirs(output_folder, exist_ok=True)
#     image_paths = []
#     try:
#         with fitz.open(pdf_path) as pdf_document:
#             for page_number, page in enumerate(pdf_document, start=1):
#                 pix = page.get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72))
#                 output_path = os.path.join(output_folder, f"page_{page_number}.png")
#                 pix.save(output_path)
#                 image_paths.append(output_path)
#     except Exception as e:
#         print(f"[ERROR] Failed to convert PDF '{os.path.basename(pdf_path)}' to images: {e}")
#         return []
#     return image_paths


# # =========================================================
# # 2. OCR with Gemini
# # =========================================================
# def ocr_with_gemini(image_paths, instruction):
#     """Extract text from images using Gemini OCR."""
#     images = []
#     try:
#         for path in image_paths:
#             with Image.open(path) as img:
#                 images.append(img.copy())

#         prompt = f"""
#         {instruction}

#         These are pages from a PDF. Extract all text content while preserving structure.
#         Pay attention to tables, columns, headers, and formatting.
#         """
#         response = model.generate_content([prompt, *images])
#         return response.text
#     except Exception as e:
#         print(f"[ERROR] OCR processing failed: {e}")
#         return ""
#     finally:
#         images.clear()


# # Retry wrapper for timeouts
# def ocr_with_retry(image_paths, instruction, retries=3, delay=5):
#     for attempt in range(1, retries + 1):
#         text = ocr_with_gemini(image_paths, instruction)
#         if text:
#             return text
#         print(f"  [RETRY] Attempt {attempt}/{retries} failed. Retrying in {delay * attempt}s...")
#         time.sleep(delay * attempt)
#     return ""


# # =========================================================
# # 3. Chunking for large PDFs
# # =========================================================
# def chunked(iterable, size):
#     for i in range(0, len(iterable), size):
#         yield iterable[i:i + size]


# def ocr_large_pdf(image_paths, instruction, chunk_size=5):
#     html_parts = []
#     for i, chunk in enumerate(chunked(image_paths, chunk_size), start=1):
#         print(f"    🔹 Processing chunk {i} ({len(chunk)} pages)...")
#         part_html = ocr_with_retry(chunk, instruction)
#         if part_html:
#             html_parts.append(part_html)
#         else:
#             print(f"    [WARNING] Chunk {i} produced no text.")
#     return "\n".join(html_parts)


# def ocr_complex_document(image_paths):
#     """Use Gemini to extract full text as HTML."""
#     instruction = (
#         "Extract the whole text from the document in HTML format while preserving structure and layout."
#         " Ensure tables and lists are properly formatted in HTML."
#         " Use appropriate HTML tags like <h1>, <h2>, <p>, <table>, <ul>, <li>, etc."
#         " Maintain paragraph breaks and spacing. No need to add styling or colors."
#         " No need for intro, المحتويات او الخاتمة, just the main content. Include the title."
#         " Don't forget we want all text in the pdf to be extracted."
#     )
#     return ocr_large_pdf(image_paths, instruction, chunk_size=10)


# # =========================================================
# # 4. Safe Cleanup
# # =========================================================
# def safe_rmtree(path, max_retries=3, delay=1):
#     """Safely remove directory tree with retries."""
#     for attempt in range(max_retries):
#         try:
#             if os.path.exists(path):
#                 import gc
#                 gc.collect()
#                 time.sleep(delay)
#                 shutil.rmtree(path)
#             return True
#         except PermissionError as e:
#             print(f"  [WARNING] Attempt {attempt + 1}/{max_retries} failed to remove temp folder: {e}")
#             if attempt < max_retries - 1:
#                 time.sleep(delay * (attempt + 1))  # Exponential backoff
#             else:
#                 print(f"  [ERROR] Could not remove temp folder after {max_retries} attempts: {path}")
#                 return False
#         except Exception as e:
#             print(f"  [ERROR] Unexpected error removing temp folder: {e}")
#             return False
#     return False


# # =========================================================
# # 5. Main
# # =========================================================
# def main():
#     pdf_input_folder = r"C:\Users\AReda\Desktop\finbot pro\data\guides"
#     output_folder = r"C:\Users\AReda\Desktop\finbot pro\tt"
#     temp_image_folder = r"C:\Users\AReda\Desktop\finbot pro\temp_images"

#     os.makedirs(output_folder, exist_ok=True)

#     pdf_files = [f for f in os.listdir(pdf_input_folder) if f.lower().endswith('.pdf')]
#     print(f"\nFound {len(pdf_files)} PDF files to process.")

#     for filename in pdf_files:
#         print(f"\n--- Processing '{filename}' ---")
#         pdf_path = os.path.join(pdf_input_folder, filename)

#         try:
#             # Step 1: Convert PDF → Images
#             print("  Step 1: Converting PDF to images...")
#             image_paths = convert_pdf_to_images(pdf_path, temp_image_folder)
#             if not image_paths:
#                 print(f"[ERROR] No images created for '{filename}'. Skipping.")
#                 continue

#             # Step 2: OCR with Gemini
#             print("  Step 2: Extracting text with Gemini AI...")
#             html_text = ocr_complex_document(image_paths)

#             if html_text:
#                 output_html_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.html")
#                 with open(output_html_path, "w", encoding="utf-8") as f:
#                     f.write(html_text)
#                 print(f"  ✅ Saved HTML to: {output_html_path}")
#             else:
#                 print(f"  [WARNING] No text extracted for '{filename}'")

#         except Exception as e:
#             print(f"[!!!] Error while processing '{filename}': {e}")
#         finally:
#             # Step 3: Cleanup
#             print("  Step 3: Cleaning up temporary files...")
#             if not safe_rmtree(temp_image_folder):
#                 try:
#                     for file_path in image_paths:
#                         if os.path.exists(file_path):
#                             os.remove(file_path)
#                     print("  ✅ Individual temp files removed")
#                 except Exception as e:
#                     print(f"  [WARNING] Could not remove individual temp files: {e}")

#     print("\n--- All PDF files processed. ---")


# if __name__ == "__main__":
#     main()
# import os
# import google.generativeai as genai
# from PIL import Image
# import base64
# import io
# import fitz  # PyMuPDF
# import os
# from dotenv import load_dotenv
# import csv
# import json
# import shutil
# import time
# import gc
# from pathlib import Path

# # ---
# # Your existing setup and functions (with improvements)
# # ---
# load_dotenv()
# api_key = os.getenv('GOOGLE_API_KEY')

# # Configure the Gemini API
# if not api_key:
#     raise ValueError("GOOGLE_API_KEY not found. Please set it in your .env file.")
# genai.configure(api_key=api_key)



# model = genai.GenerativeModel(
#     'gemini-2.5-pro',  # Changed to 1.5-pro for better stability
# )

# def convert_pdf_to_images(pdf_path, output_folder, dpi=120):  # Reduced DPI for faster processing
#     """Converts each page of a PDF to a PNG image."""
#     os.makedirs(output_folder, exist_ok=True)
#     image_paths = []
#     try:
#         pdf_document = fitz.open(pdf_path)
#         for page_number in range(len(pdf_document)):
#             page = pdf_document[page_number]
#             # Reduced resolution for faster processing and smaller file sizes
#             pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
#             output_path = os.path.join(output_folder, f"page_{page_number + 1}.png")
#             pix.save(output_path)
#             image_paths.append(output_path)
#         pdf_document.close()
#     except Exception as e:
#         print(f"  [ERROR] Failed to convert PDF '{os.path.basename(pdf_path)}' to images: {e}")
#         return []
#     return image_paths

# def safe_cleanup_images(image_paths, max_retries=3):
#     """Safely cleanup image files with retry logic."""
#     for image_path in image_paths:
#         if os.path.exists(image_path):
#             for attempt in range(max_retries):
#                 try:
#                     os.remove(image_path)
#                     break
#                 except PermissionError:
#                     print(f"  [WARNING] Could not delete {image_path} (attempt {attempt + 1}), retrying...")
#                     time.sleep(1)  # Wait a bit before retrying
#                     gc.collect()  # Force garbage collection
#                 except Exception as e:
#                     print(f"  [ERROR] Failed to delete {image_path}: {e}")
#                     break

# def safe_cleanup_folder(folder_path, max_retries=3):
#     """Safely cleanup temporary folder with retry logic."""
#     if os.path.exists(folder_path):
#         for attempt in range(max_retries):
#             try:
#                 shutil.rmtree(folder_path)
#                 break
#             except PermissionError:
#                 print(f"  [WARNING] Could not delete folder {folder_path} (attempt {attempt + 1}), retrying...")
#                 time.sleep(2)
#                 gc.collect()
#             except Exception as e:
#                 print(f"  [ERROR] Failed to delete folder {folder_path}: {e}")
#                 break

# def ocr_with_gemini_retry(image_paths, instruction, max_retries=3, base_delay=5):
#     """Processes images with Gemini OCR with retry logic for timeout errors."""
    
#     # Close PIL images properly to avoid file locks
#     images = []
#     try:
#         for path in image_paths:
#             img = Image.open(path)
#             # Convert to bytes to avoid file locks
#             img_byte_arr = io.BytesIO()
#             img.save(img_byte_arr, format='PNG')
#             img_byte_arr.seek(0)
#             images.append(Image.open(img_byte_arr))
#             img.close()  # Close the original file handle
#     except Exception as e:
#         print(f"  [ERROR] Failed to load images: {e}")
#         return None
    
#     prompt = f"""
#     {instruction}
    
#     These are pages from a PDF document. Extract all text content while preserving the structure.
#     Pay special attention to tables, columns, headers, and any structured content.
#     Maintain paragraph breaks and formatting.
#     """
    
#     for attempt in range(max_retries):
#         try:
#             print(f"  Attempt {attempt + 1} to process with Gemini...")
#             response = model.generate_content([prompt, *images])
#             return response.text
            
#         except Exception as e:
#             error_msg = str(e).lower()
#             if "504" in error_msg or "deadline" in error_msg or "timeout" in error_msg:
#                 if attempt < max_retries - 1:
#                     delay = base_delay * (2 ** attempt)  # Exponential backoff
#                     print(f"  [WARNING] Timeout error (attempt {attempt + 1}). Retrying in {delay} seconds...")
#                     time.sleep(delay)
#                     continue
#                 else:
#                     print(f"  [ERROR] Max retries reached for timeout error: {e}")
#                     return None
#             elif "quota" in error_msg or "rate" in error_msg:
#                 if attempt < max_retries - 1:
#                     delay = 60  # Wait longer for quota issues
#                     print(f"  [WARNING] Rate limit/quota error. Waiting {delay} seconds...")
#                     time.sleep(delay)
#                     continue
#                 else:
#                     print(f"  [ERROR] Max retries reached for rate limit: {e}")
#                     return None
#             else:
#                 print(f"  [ERROR] Unexpected error: {e}")
#                 return None
    
#     return None

# def ocr_complex_document(image_paths):
#     """Creates the specific prompt and calls the Gemini OCR function with retry."""
#     instruction = """
#         اقرأ النص القانوني أو حكم اللجنة المرفق بعناية تامة، واستخرج تفصيلاً كاملاً للقرار وفق الشكل التالي. التزم بالقواعد والتعليمات أدناه حرفياً.

#         التزامات عامة:
#         - استند فقط إلى النص الأصلي كما ورد في المستند؛ لا تضف أي تفسير، استنتاج، أو معلومات من خارج النص.
#         - استخدم اللغة العربية الفصحى في كل الحقول.
#         - لا تذكر أي مواد قانونية أو أرقام مواد/مراجع نظامية.
#         - لا تذكر أي مبالغ مالية أو أرقام نقدية في أي حقل.
#         - إذا كان أي حقل غير موجود صراحة في النص، اتركه كسلسلة فارغة "".
#         - احتفظ بأسلوب قانوني ورسمي، لكن صِف بوضوح وبالتفصيل ما ورد في القرار عن وجهات النظر والأسباب.
#         - إذا كان النص المستخرج غير واضح أو به غموض في المعنى، انسخه كما هو دون تعديل أو تفسير أو إعادة صياغة لتجنّب أي سوء فهم.
#         - لا تضف أي نص خارج بنية JSON النهائية.

#         هيكل الإخراج (أعد النتيجة بنفس البنية أدناه تمامًا):

#         {
#         "تفصيل_القرار": {
#             "رقم_القرار_النهائي": "استخرج رقم القرار النهائي كما ورد نصاً. يكون موجود في رأس أول صفحة ويبدأ بـ IR أو VA.",
#             "اسم_الدائرة_الابتدائية": "استخرج اسم دائرة الفصل الابتدائية كما ورد في النص.",
#             "اسم_الدائرة_النهائية": "استخرج اسم الدائرة الاستئنافية أو اللجنة النهائية كما ورد في النص.",
#             "اللجنة_الابتدائية": "سرد تفصيلي لقرار اللجنة الابتدائية كما ورد نصاً، متضمناً الأسباب والمبررات التي استندت إليها اللجنة ونتيجة القرار في هذه المرحلة.",
#             "اللجنة_النهائية": "سرد تفصيلي لقرار اللجنة النهائية كما ورد نصاً، متضمناً الأسباب والمبررات التي استندت إليها اللجنة ونتيجة القرار في هذه المرحلة.",
#             "البنود_محل_الدعوى": [
#             {
#                 "اسم_البند": "انسخ اسم وعبارة البند كما وردت في القرار (عنوان البند).",
#                 "نبذة_مختصرة_عن_الاعتراض": "خلاصة وجيزة (جملة أو اثنتان) تشرح طبيعة الاعتراض على هذا البند كما وردت في القرار.",
#                 "وجهة_نظر_المكلف_بالتفصيل": "انقل بالتفصيل كل ما ورد في نص القرار عن دفوع المكلف/الشركة بهذا البند: الحجج، الوقائع والمستندات التي استند إليها، المطالبات والإشارات الزمنية أو العقدية أو المحاسبية أو الفنية التي ذكرها المكلف. انسخ النص قدر الإمكان مع إعادة ترتيب بسيط لقراءة منطقية إذا لزم، لكن لا تُحوّل المعنى ولا تبتّ من النص الأصلي.",
#                 "وجهة_نظر_الهيئة_بالتفصيل": "انقل بالتفصيل كل ما ورد في نص القرار عن موقف الهيئة بهذا البند: التبريرات، الأدلة أو النتائج الرقابية، طريقة تفسير الهيئة للوقائع والعقود والسجلات، وأي حجج فنية أو إجرائية ذكرتها الهيئة.",
#                 "رأي_اللجنة_الابتدائية_ومبرراته": "انقل نص قرار اللجنة الابتدائية في هذا البند مع الأسباب والمبررات التي استندت إليها اللجنة. اذكر تفاصيل القرار ونتيجته كما وردت نصاً.وفي حاله عدم توفرها بشكل صريح قم باستنتاجها بناء علي المعطيات",
#                 "رأي_اللجنة_النهائية_ومبرراته": "انقل نص قرار اللجنة النهائية في هذا البند مع الأسباب والمبررات التي استندت إليها اللجنة. لا تختصر، وأعد الأسباب كاملة أو بنحو قريب جداً مع الحفاظ على المعنى والتفصيل."
#             }
#             ],
#             "خلاصة_نهائية": "جملة أو فقرة واحدة تُلخّص نتيجة القرار الكلية كما وردت (مثل: قبول الاستئناف شكلاً وإعادة الدعوى إلى دائرة الفصل للنظر في بنود محددة)، لكن بدون ذكر مواد قانونية أو مبالغ."
#         }
#         }

#         قواعد تنفيذية إضافية:
#         1. لا تُدرج في أي حقل عناوين مواد قانونية أو أرقام مواد أو نصوص نظامية.
#         2. في حالة عدم ذكر قرار اللجنة الابتدائية قم باستنتاجه بناء على السياق.
#         3. لا تذكر مبالغ مالية، ولا قيّم الخسائر أو التزامات نقدية.
#         4. عند نقل وجهات النظر والأسباب، أذكر بوضوح ما استندت إليه كل جهة (مثلاً: "استندت الشركة إلى القوائم المالية المدققة والعقود المبرمة" أو "استندت الهيئة إلى نتائج الفحص المستندي") — لكن هذا فقط إذا ورد صراحة في النص.
#         5. حافظ على الترتيب كما في القالب؛ لا تضف حقولًا جديدة ولا تزيل الحقول المطلوبة.
#         6. أعد الناتج بصيغة JSON صالحة فقط ولا تضف أي تعليقات خارجها.
#         7. قم بسرد تفصيل لقرار (اللجنة الابتدائية) في البداية وأذكر اسم اللجنة من واقع القرار.
#         8. قم بسرد تفصيل لقرار (اللجنة النهائية) في البداية وأذكر اسم اللجنة من واقع القرار.
#         9. لكل بند من البنود محل الدعوى، يجب أن يحتوي على (رأي ابتدائي) و(رأي نهائي) مستقلين مع المبررات كما وردت في النص.
#         10. إذا كان النص المستخرج غير واضح أو به التباس، انسخه كما هو دون تعديل أو تفسير لتجنّب أي سوء فهم.
#         """

#     return ocr_with_gemini_retry(image_paths, instruction)

# def clean_gemini_output(raw_text):
#     """Cleans the raw text from Gemini to extract the JSON part."""
#     if not raw_text:
#         return "{}"
    
#     # Find the start and end of the JSON block
#     start_index = raw_text.find('{')
#     end_index = raw_text.rfind('}')
    
#     if start_index != -1 and end_index != -1:
#         json_str = raw_text[start_index:end_index+1]
#         return json_str
    
#     # Return empty JSON if no valid block is found
#     return "{}"

# def save_progress(results, output_file):
#     """Save current progress to avoid losing data."""
#     try:
#         with open(output_file, "w", encoding="utf-8") as f:
#             json.dump(results, f, ensure_ascii=False, indent=2)
#         print(f"  ✅ Progress saved to {output_file}")
#     except Exception as e:
#         print(f"  [ERROR] Could not save progress: {e}")

# def main():
#     """
#     Main function to orchestrate the PDF processing workflow.
#     Saves all extracted data into a JSON file with progress saving.
#     """
#     # --- CONFIGURATION ---
#     pdf_input_folder = r"C:\Users\AReda\Desktop\finbot pro\ICT-WHT\Decesions\ICT-WHT"
#     temp_image_folder = r"C:\Users\AReda\Desktop\finbot pro\ICT-WHT\temp_images"
#     output_json_file = r"C:\Users\AReda\Desktop\finbot pro\ICT-WHT\decisions-ICT-WHT.json"
#     progress_file = r"C:\Users\AReda\Desktop\finbot pro\ICT-WHT\progress_backup-ICT-WHT.json"
#     # --- END CONFIGURATION ---

#     results = []  # list of extracted dicts
#     processed_files = set()  # Track processed files

#     # Load existing progress if available
#     if os.path.exists(progress_file):
#         try:
#             with open(progress_file, "r", encoding="utf-8") as f:
#                 results = json.load(f)
#                 processed_files = {item.get('Source_Filename', '') for item in results}
#             print(f"Loaded {len(results)} previously processed files from progress backup.")
#         except Exception as e:
#             print(f"Could not load progress file: {e}")

#     # Get PDF files
#     pdf_files = [f for f in os.listdir(pdf_input_folder) if f.lower().endswith('.pdf')]
#     remaining_files = [f for f in pdf_files if f not in processed_files]
    
#     print(f"\nFound {len(pdf_files)} total PDF files.")
#     print(f"Already processed: {len(processed_files)}")
#     print(f"Remaining to process: {len(remaining_files)}")

#     for i, filename in enumerate(remaining_files, 1):
#         print(f"\n--- Processing '{filename}' ({i}/{len(remaining_files)}) ---")
#         pdf_path = os.path.join(pdf_input_folder, filename)
#         image_paths = []

#         try:
#             # 1. Convert PDF to images
#             print("  Step 1: Converting PDF to images...")
#             image_paths = convert_pdf_to_images(pdf_path, temp_image_folder)

#             if not image_paths:
#                 print(f"  [ERROR] No images were created for '{filename}'. Skipping.")
#                 continue

#             # 2. Extract data with Gemini (with retry logic)
#             print("  Step 2: Extracting text with Gemini AI...")
#             raw_gemini_text = ocr_complex_document(image_paths)

#             if raw_gemini_text is None:
#                 print(f"  [ERROR] Failed to extract text for '{filename}'. Skipping.")
#                 continue

#             # 3. Parse JSON output
#             print("  Step 3: Parsing AI output...")
#             json_text = clean_gemini_output(raw_gemini_text)
            
#             try:
#                 extracted_data = json.loads(json_text)
#             except json.JSONDecodeError as e:
#                 print(f"  [ERROR] Invalid JSON output for '{filename}': {e}")
#                 print(f"  Raw output: {json_text[:200]}...")
#                 continue

#             # Add source filename
#             extracted_data['Source_Filename'] = filename

#             # 4. Append to results list
#             results.append(extracted_data)

#             print(f"  ✅ Successfully processed '{filename}'")
            
#             # Save progress every 5 files
#             if len(results) % 5 == 0:
#                 save_progress(results, progress_file)

#         except Exception as e:
#             print(f"  [!!!] Unexpected error while processing '{filename}': {e}")
#             continue

#         finally:
#             # Cleanup temp images more safely
#             if image_paths:
#                 print("  Cleaning up temporary files...")
#                 safe_cleanup_images(image_paths)
#                 time.sleep(1)  # Give system time to release file handles
            
#             # Clean up temp folder if it exists
#             safe_cleanup_folder(temp_image_folder)

#         # Add a small delay between files to avoid rate limiting
#         if i < len(remaining_files):
#             print("  Waiting before next file...")
#             time.sleep(2)

#     print("\n--- All PDF files have been processed. ---")

#     # Save final results
#     try:
#         with open(output_json_file, "w", encoding="utf-8") as f:
#             json.dump(results, f, ensure_ascii=False, indent=2)
#         print(f"✅ Saved extracted data to {output_json_file}")
        
#         # Clean up progress file
#         if os.path.exists(progress_file):
#             os.remove(progress_file)
            
#     except Exception as e:
#         print(f"[ERROR] Could not save final JSON file: {e}")

#     return results


# # Example usage
# if __name__ == "__main__":
#     extracted_chunks = main()
#     print(f"\nProcessing complete! Total extracted documents: {len(extracted_chunks)}")
#     if extracted_chunks:
#         print("\nSample Extracted Data (first item):")
#         print(json.dumps(extracted_chunks[0], ensure_ascii=False, indent=2))

In [10]:
import os
import json

# المسار اللي فيه ملفات الـ JSON
input_dir = r"C:\Users\AReda\Desktop\finbot pro\FinBot\Data\ICT-WHT\Guidelines_json"

# مسار ملف الإخراج
output_file = r"C:\Users\AReda\Desktop\finbot pro\FinBot\Data\ICT-WHT\combined_Guidelines.json"

combined_data = []

# المرور على كل الملفات داخل الفولدر
for filename in os.listdir(input_dir):
    if filename.endswith(".json"):
        file_path = os.path.join(input_dir, filename)
        
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            # لو الملف عبارة عن list
            if isinstance(data, list):
                combined_data.extend(data)
            else:
                combined_data.append(data)

# حفظ الملف النهائي
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(combined_data, f, ensure_ascii=False, indent=2)

print(f"Combined {len(combined_data)} records into:")
print(output_file)


Combined 39 records into:
C:\Users\AReda\Desktop\finbot pro\FinBot\Data\ICT-WHT\combined_Guidelines.json


In [13]:
import os
import json

BASE_DIR = r"C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data"

def extract_source_type(filename):
    """
    combined_Guidelines.json -> Guidelines
    combined_Laws.json -> Laws
    """
    name = os.path.splitext(filename)[0]
    return name.replace("combined_", "")

for folder_name in os.listdir(BASE_DIR):
    folder_path = os.path.join(BASE_DIR, folder_name)

    if not os.path.isdir(folder_path):
        continue

    for file_name in os.listdir(folder_path):
        if not file_name.lower().endswith(".json"):
            continue

        file_path = os.path.join(folder_path, file_name)
        source_type = extract_source_type(file_name)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # تأكد أن الملف List
        if not isinstance(data, list):
            continue

        for item in data:
            item["category"] = folder_name        # ICT-WHT / VAT
            item["source_type"] = source_type     # Guidelines / Laws

        # overwrite نفس الملف (تقدر تغير الاسم لو حابب)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"Updated: {file_path}")

Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\ICT-WHT\combined_Guidelines.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\ICT-WHT\combined_Laws.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\ICT-WHT\combined_publications.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\ICT-WHT\decisions-ICT-WHT.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\ICT-WHT\decisions-ZA-ICT-WHT.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\VAT\combind_decisions.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\VAT\combined_guidelines.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\VAT\combined_laws.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\VAT\combined_publications.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\ZA-1440\combined_decisions.json
Updated: C:\Users\AReda\Desktop\finbot pro\FinBot\Data\data\ZA-1440\combined_guidelines.json
Updat

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

response = client.responses.create(
    model="gpt-4.1",
    input="Write a one-sentence bedtime story about a unicorn."
)

# Print the model's text output (convenience property)
print(response.output_text)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}